In [1]:
# ============================================================
# CELL 1.1 — Environment Verification
#
# Installation is handled OUTSIDE the notebook via:
#   uv sync --extra parsers        (fast parsers only)
#   uv sync --all-extras           (full benchmark)
#
# This cell verifies you're running inside the uv-managed venv
# and that the core dependencies are importable. If something
# is missing, it tells you the exact uv sync command to fix it.
# ============================================================

import sys
import gc
from pathlib import Path

# ---- 1. Confirm we're inside the uv venv, not system Python ----
venv_path = Path(sys.prefix)
in_venv = venv_path.name == ".venv" or (venv_path / "pyvenv.cfg").exists()

if not in_venv:
    print(
        "⚠ WARNING: You don't appear to be running inside the uv venv.\n"
        "  Launch the notebook with:\n"
        "    uv run jupyter notebook notebooks/benchmark.ipynb\n"
        "  This ensures the correct interpreter and all installed packages are used."
    )
else:
    print(f"✓ Running inside venv: {venv_path}")

print(f"  Python {sys.version}")
print(f"  Executable: {sys.executable}\n")

# ---- 2. Verify core dependencies ----
# These are in [project.dependencies] — always present after any uv sync.
CORE_DEPS = [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("sklearn", "scikit-learn"),
    ("sentence_transformers", "sentence-transformers"),
    ("qdrant_client", "qdrant-client"),
    ("rouge_score", "rouge-score"),
    ("tqdm", "tqdm"),
    ("rich", "rich"),
    ("dotenv", "python-dotenv"),
    ("chonkie", "chonkie[semantic]"),
    ("langchain_text_splitters", "langchain-text-splitters"),
]

missing_core = []
for import_name, pip_name in CORE_DEPS:
    try:
        __import__(import_name)
        print(f"  ✓ {pip_name}")
    except ImportError:
        print(f"  ✗ {pip_name}  ← MISSING")
        missing_core.append(pip_name)

if missing_core:
    print(
        f"\n✗ {len(missing_core)} core dependencies missing. Run:\n"
        "    uv sync\n"
        "  then restart the kernel."
    )
else:
    print("\n✓ All core dependencies present.")

# ---- 3. Check optional parser groups ----
OPTIONAL_DEPS = {
    "parsers": [
        ("fitz", "pymupdf"),
        ("pdfplumber", "pdfplumber"),
        ("openparse", "openparse"),
    ],
    "heavy": [
        ("docling", "docling[pdf]"),
        ("marker", "marker-pdf"),
        ("unstructured", "unstructured[pdf]"),
    ],
    "cloud": [
        ("llama_parse", "llama-parse"),
    ],
}

print("\n── Optional parser groups ──")
for group, deps in OPTIONAL_DEPS.items():
    available = []
    missing = []
    for import_name, pip_name in deps:
        try:
            __import__(import_name)
            available.append(pip_name)
        except ImportError:
            missing.append(pip_name)

    status = "✓" if not missing else "partial" if available else "✗"
    print(f"  [{status}] --extra {group}")
    for name in available:
        print(f"        ✓ {name}")
    for name in missing:
        print(f"        ✗ {name}  ← uv sync --extra {group}")

✓ Running inside venv: /home/dali/pdf-parser-benchmark/.venv
  Python 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
  Executable: /home/dali/pdf-parser-benchmark/.venv/bin/python

  ✓ numpy
  ✓ pandas
  ✓ matplotlib
  ✓ seaborn
  ✓ scikit-learn
  ✓ sentence-transformers
  ✓ qdrant-client
  ✓ rouge-score
  ✓ tqdm
  ✓ rich
  ✓ python-dotenv
  ✓ chonkie[semantic]
  ✓ langchain-text-splitters

✓ All core dependencies present.

── Optional parser groups ──
  [✓] --extra parsers
        ✓ pymupdf
        ✓ pdfplumber
        ✓ openparse
  [✓] --extra heavy
        ✓ docling[pdf]
        ✓ marker-pdf
        ✓ unstructured[pdf]
  [✗] --extra cloud
        ✗ llama-parse  ← uv sync --extra cloud


In [2]:
# Non-parser imports — always required
import subprocess
from typing import Optional
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from enum import Enum
import seaborn as sns
from tqdm import tqdm
from rich.console import Console
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer
from chonkie import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
console = Console()

# Parser availability — read by BaseParser.is_available()
# Actual import happens inside each parser's _extract() method
# so a missing parser never breaks the notebook on import
PARSER_AVAILABLE: dict[str, bool] = {}

for parser_name, import_name in [
    ("pymupdf", "fitz"),
    ("pdfplumber", "pdfplumber"),
    ("docling", "docling"),
    ("unstructured", "unstructured"),
    ("openparse", "openparse"),
    ("llamaparse", "llama_parse"),
]:
    try:
        __import__(import_name)
        PARSER_AVAILABLE[parser_name] = True
    except ImportError:
        PARSER_AVAILABLE[parser_name] = False

# LiteParse — Node.js availability check
try:
    __import__("liteparse")
    PARSER_AVAILABLE["liteparse"] = True
except ImportError:
    PARSER_AVAILABLE["liteparse"] = False

console.print(PARSER_AVAILABLE)

{
    'pymupdf': True,
    'pdfplumber': True,
    'docling': True,
    'unstructured': True,
    'openparse': True,
    'llamaparse': False,
    'liteparse': True
}

In [ ]:
from dataclasses import dataclass, field, asdict

# ---- Global config ----
PDF_PATH = Path("/home/dali/pdf-parser-benchmark/data/Jeanne Boyarsky, Scott Selikoff - OCP Oracle Certified Professional Java SE 17 Developer Study Guide - Exam 1Z0-829.-Sybex (2022).pdf")
CACHE_DIR = Path("./benchmark_cache")
RESULTS_DIR = Path("./benchmark_results")
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
EMBEDDING_DIM = 384
QDRANT_COLLECTION_PREFIX = "benchmark_"
SEMANTIC_CHUNK_SIZE = 1024
RECURSIVE_CHUNK_SIZE = 1000
RECURSIVE_CHUNK_OVERLAP = 150

CACHE_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ---- Result types ----

@dataclass
class ExtractionResult:
    """
    Output of running one parser on the full PDF.
    
    raw_text: Concatenated full text, newline-separated by page.
              Used for chunking and chunk quality evaluation.
    pages:    Per-page text list. Index 0 = page 1.
              Needed to compute page-level code block density.
    metadata: Parser-specific structured output (e.g., Docling's
              DoclingDocument, openparse's Node list). Stored as
              a list of dicts for uniform downstream handling.
    extraction_time_s: Wall-clock seconds for the full parse.
    parser_name: Matches the key in PARSER_AVAILABLE.
    error: Non-None if the parser raised during extraction.
           We store it rather than re-raising so the loop continues.
    """
    parser_name: str
    raw_text: str = ""
    pages: list[str] = field(default_factory=list)
    metadata: list[dict] = field(default_factory=list)
    extraction_time_s: float = 0.0
    error: Optional[str] = None

@dataclass
class Chunk:
    """
    A single chunk produced by one (parser, chunker) combination.
    
    chunk_id:    Stable identifier: "{parser}_{chunker}_{index:04d}"
    text:        The chunk text.
    parser:      Which parser produced the source text.
    chunker:     "semantic" | "recursive"
    char_count / token_count: Raw size metrics.
    starts_mid_sentence: Heuristic — does the chunk start with a
                         lowercase letter? Indicates a bad split.
    ends_mid_sentence:   Does the chunk end without terminal punctuation?
    contains_code_block: Heuristic — does the chunk contain a Java
                         code pattern (braces, semicolons, keywords)?
    code_block_fragmented: True if the chunk contains an *unclosed*
                           code block (opening brace without closing).
    """
    chunk_id: str
    text: str
    parser: str
    chunker: str
    char_count: int = 0
    token_count: int = 0
    starts_mid_sentence: bool = False
    ends_mid_sentence: bool = False
    contains_code_block: bool = False
    code_block_fragmented: bool = False

@dataclass
class ChunkQualityReport:
    """
    Aggregate quality metrics for one (parser, chunker) pair.
    
    total_chunks:           How many chunks were produced.
    avg_chunk_size_chars:   Mean chunk length in characters.
    code_block_preservation_rate: Fraction of detected code blocks
                             that are intact (not fragmented).
    mid_sentence_start_rate: Fraction of chunks starting mid-sentence.
    mid_sentence_end_rate:   Fraction of chunks ending mid-sentence.
    avg_rouge_l:             Mean ROUGE-L vs. gold reference passages
                             (computed in Cell 4).
    """
    parser: str
    chunker: str
    total_chunks: int = 0
    avg_chunk_size_chars: float = 0.0
    code_block_preservation_rate: float = 0.0
    mid_sentence_start_rate: float = 0.0
    mid_sentence_end_rate: float = 0.0
    avg_rouge_l: float = 0.0
    avg_bleu: float = 0.0
    avg_semantic_sim: float = 0.0

@dataclass
class RetrievalResult:
    """
    Retrieval evaluation for one (parser, chunker) pair on one query.
    
    query_id:          Index into the gold query set.
    query_text:        The query string.
    retrieved_chunk_ids: Top-K chunk IDs returned by Qdrant.
    relevant_chunk_ids:  Gold-annotated relevant chunk IDs for this query.
    precision_at_k:    |retrieved ∩ relevant| / k
    recall_at_k:       |retrieved ∩ relevant| / |relevant|
    """
    parser: str
    chunker: str
    query_id: int
    query_text: str
    retrieved_chunk_ids: list[str] = field(default_factory=list)
    relevant_chunk_ids: list[str] = field(default_factory=list)
    precision_at_k: float = 0.0
    recall_at_k: float = 0.0

# ---- LiteParse Node.js runner scaffold ----


# ---- Sanity check: PDF exists ----
if not PDF_PATH.exists():
    console.print(
        f"[bold red]⚠ PDF not found at '{PDF_PATH}'. "
        "Update PDF_PATH before running the extraction cells.[/bold red]"
    )
else:
    size_mb = PDF_PATH.stat().st_size / 1_048_576
    console.print(f"[green]✓ PDF found:[/green] {PDF_PATH.name} ({size_mb:.1f} MB)")

console.print("\n[bold]Configuration[/bold]")
console.print(f"  Embedding model : {EMBEDDING_MODEL_NAME}")
console.print(f"  Semantic chunk size : {SEMANTIC_CHUNK_SIZE} tokens")
console.print(f"  Recursive chunk size: {RECURSIVE_CHUNK_SIZE} chars (overlap: {RECURSIVE_CHUNK_OVERLAP})")
console.print(f"  Cache dir  : {CACHE_DIR}")
console.print(f"  Results dir: {RESULTS_DIR}")

✓ PDF found: Jeanne Boyarsky, Scott Selikoff - OCP Oracle Certified Professional Java SE 17 Developer Study Guide -
Exam 1Z0-829.-Sybex (2022).pdf (19.1 MB)

Configuration

Embedding model : BAAI/bge-small-en-v1.5

Semantic chunk size : 1024 tokens

Recursive chunk size: 1000 chars (overlap: 150)

Cache dir  : benchmark_cache

Results dir: benchmark_results

In [4]:
import hashlib
import json
from abc import ABC, abstractmethod

class BaseParser(ABC):
    """
    Abstract base class for all PDF parsers in this benchmark.
    
    Subclasses implement _extract() with their parser-specific logic.
    The public extract() method wraps _extract() with:
      - Cache check (load from disk if already parsed)
      - Wall-clock timing
      - Error isolation (exceptions stored in result.error, not raised)
    """

    def __init__(self, cache_dir: Path = CACHE_DIR):
        self.cache_dir = cache_dir
        self.name = self.__class__.__name__.replace("Parser", "").lower()

    def _pdf_hash(self, pdf_path: Path) -> str:
        """
        SHA-256 of the first 64KB of the PDF.
        
        We don't hash the entire file (could be hundreds of MB) — the
        first 64KB is enough to detect if the file changed between runs.
        Collisions here would just mean a stale cache hit, not a
        correctness issue.
        """
        h = hashlib.sha256()
        with open(pdf_path, "rb") as f:
            h.update(f.read(65536))
        return h.hexdigest()[:16]  # 16 hex chars is plenty for a cache key

    def _cache_path(self, pdf_path: Path) -> Path:
        pdf_hash = self._pdf_hash(pdf_path)
        return self.cache_dir / f"{self.name}_{pdf_hash}.json"

    def _load_cache(self, pdf_path: Path) -> Optional[ExtractionResult]:
        cache_path = self._cache_path(pdf_path)
        if not cache_path.exists():
            return None
        try:
            with open(cache_path) as f:
                data = json.load(f)
            result = ExtractionResult(**data)
            # Don't replay failures — always retry failed parsers
            if result.error:
                console.print(f"  [yellow]Previous run failed for {self.name} — retrying[/yellow]")
                return None
            console.print(f"  [dim]Cache hit for {self.name} — loading from {cache_path.name}[/dim]")
            return result
        except Exception as e:
            console.print(f"  [yellow]Cache read failed for {self.name}: {e} — re-extracting[/yellow]")
            return None

    def _save_cache(self, result: ExtractionResult, pdf_path: Path) -> None:
        cache_path = self._cache_path(pdf_path)
        try:
            def default_serializer(obj):
                if isinstance(obj, set):
                    return list(obj)
                if isinstance(obj, Enum):
                    return obj.value
                raise TypeError(f"Object of type {type(obj)} is not JSON serializable")
    
            with open(cache_path, "w") as f:
                json.dump(asdict(result), f, ensure_ascii=False, default=default_serializer)
        except Exception as e:
            console.print(f"  [yellow]Cache write failed for {self.name}: {e}[/yellow]")

    def extract(self, pdf_path: Path) -> ExtractionResult:
        """
        Public entry point. Handles caching, timing, and error isolation.
        Subclasses implement _extract() instead of overriding this.
        """
        # Cache check
        cached = self._load_cache(pdf_path)
        if cached is not None:
            return cached

        console.print(f"\n[bold cyan]→ Running {self.name}...[/bold cyan]")
        start = time.perf_counter()

        try:
            result = self._extract(pdf_path)
            result.extraction_time_s = time.perf_counter() - start
            result.parser_name = self.name
        except Exception as e:
            result = ExtractionResult(
                parser_name=self.name,
                extraction_time_s=time.perf_counter() - start,
                error=f"{type(e).__name__}: {e}",
            )
            console.print(f"  [red]✗ {self.name} failed: {result.error}[/red]")

        self._save_cache(result, pdf_path)
        return result

    @abstractmethod
    def _extract(self, pdf_path: Path) -> ExtractionResult:
        """
        Parser-specific extraction logic.
        Must return an ExtractionResult with at minimum raw_text populated.
        extraction_time_s and parser_name are set by the base class.
        """
        ...

    def is_available(self) -> bool:
        """
        Returns True if this parser's dependencies are installed.
        Checked before running — unavailable parsers are skipped cleanly.
        """
        return PARSER_AVAILABLE.get(self.name, False)

In [5]:
import pdfplumber

class PDFPlumberParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:

        pages = []
        metadata_tables = []

        with pdfplumber.open(str(pdf_path)) as pdf:
            for page_num, page in enumerate(tqdm(pdf.pages, desc="pdfplumber pages")):

                # Extract tables first
                tables = page.extract_tables()
                table_texts = []
                table_bboxes = []

                for table in (tables or []):
                    if not table:
                        continue
                    rows = []
                    for row in table:
                        cells = [str(cell or "").strip() for cell in row]
                        rows.append("| " + " | ".join(cells) + " |")
                    table_text = "\n".join(rows)
                    table_texts.append(table_text)
                    metadata_tables.append({
                        "page": page_num,
                        "rows": len(table),
                        "cols": len(table[0]) if table else 0,
                        "text": table_text,
                    })

                # Crop out table bounding boxes before extracting prose
                # to avoid duplicate content
                cropped_page = page
                for table_obj in page.find_tables():
                    cropped_page = cropped_page.outside_bbox(table_obj.bbox)

                page_text = cropped_page.extract_text(
                    x_tolerance=3,
                    y_tolerance=3,
                ) or ""

                # Combine prose + tables
                full_page = page_text
                if table_texts:
                    full_page += "\n\n" + "\n\n".join(table_texts)

                pages.append(full_page)

        raw_text = "\n".join(pages)

        return ExtractionResult(
            parser_name=self.name,
            raw_text=raw_text,
            pages=pages,
            metadata=metadata_tables,
        )

In [6]:
import pymupdf

class PyMuPDFParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:
          # PyMuPDF

        doc = pymupdf.open(str(pdf_path))
        pages = []
        metadata_blocks = []

        for page_num, page in enumerate(tqdm(doc, desc="PyMuPDF pages")):
            # "text" mode: respects reading order, joins words into lines
            page_text = page.get_text("text")
            pages.append(page_text)

            # "blocks" mode: returns (x0, y0, x1, y1, text, block_no, block_type)
            # block_type 0 = text, 1 = image
            # We store block-level data for the code block analysis in Cell 4.
            blocks = page.get_text("blocks")
            for block in blocks:
                if block[6] == 0:  # text block only
                    metadata_blocks.append({
                        "page": page_num,
                        "bbox": block[:4],
                        "text": block[4],
                        # Font info requires iterating spans — we do a lighter
                        # heuristic here: check if text contains Java code patterns
                        "looks_like_code": _heuristic_is_code(block[4]),
                    })

        doc.close()
        raw_text = "\n".join(pages)

        return ExtractionResult(
            parser_name=self.name,
            raw_text=raw_text,
            pages=pages,
            metadata=metadata_blocks,
        )


def _heuristic_is_code(text: str) -> bool:
    """
    Lightweight heuristic to detect Java code blocks in extracted text.
    
    We look for a combination of signals rather than any single one,
    because each signal alone produces too many false positives:
    - Semicolons at line endings → Java statements
    - Curly braces → Java blocks
    - Java keywords at line start → method/class declarations
    - High punctuation density → code vs prose ratio
    
    This heuristic is used across ALL parsers for a fair comparison.
    A parser that preserves code blocks well will have these signals
    concentrated in dedicated chunks rather than mixed with prose.
    """
    if not text or len(text.strip()) < 20:
        return False

    lines = text.strip().splitlines()
    if not lines:
        return False

    # Signal 1: semicolons ending lines (Java statements)
    semicolon_lines = sum(1 for l in lines if l.rstrip().endswith(";"))

    # Signal 2: curly brace lines
    brace_lines = sum(1 for l in lines if "{" in l or "}" in l)

    # Signal 3: Java keywords at start of (stripped) line
    java_keywords = {
        "public", "private", "protected", "class", "interface",
        "void", "int", "String", "boolean", "return", "import",
        "for", "while", "if", "else", "try", "catch", "new",
        "static", "final", "extends", "implements",
    }
    keyword_lines = sum(
        1 for l in lines
        if l.strip().split(" ")[0] in java_keywords
    )

    total_lines = max(len(lines), 1)
    code_signal = (semicolon_lines + brace_lines + keyword_lines) / total_lines

    # Threshold: >30% of lines showing code signals → treat as code
    return code_signal > 0.3

In [7]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
import torch

from docling.utils.model_downloader import download_models
from pathlib import Path



class DoclingParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:
        
        download_models(
    output_dir=Path.home() / ".cache/docling/models",
    with_layout=True,
    with_tableformer=True,
    with_easyocr=False,        # we don't need OCR
    with_code_formula=False,
    with_picture_classifier=False,
)

        # PdfPipelineOptions controls which sub-models run.
        # do_table_structure=True: enables table cell reconstruction
        # do_ocr=False: we have a text-layer PDF, OCR would slow things down
        pipeline_options = PdfPipelineOptions(
            do_table_structure=True,
            do_ocr=False,
        )

        converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

        console.print("  [dim]Running Docling conversion (this will take a while)...[/dim]")
        result_doc = converter.convert(str(pdf_path))

        # Export to Markdown — this is the format we use for chunking.
        # Docling's Markdown export uses:
        #   - ## for section headings (detected by font size/style)
        #   - ``` fences for code blocks (detected by layout model)
        #   - | pipes for tables
        markdown_text = result_doc.document.export_to_markdown()

        # Also store the structured element list for Cell 4 analysis.
        # Each element has a label (TEXT, SECTION_HEADER, TABLE, CODE, etc.)
        metadata_elements = []
        for element, _ in result_doc.document.iterate_items():
            metadata_elements.append({
                "label": str(element.label) if hasattr(element, "label") else "unknown",
                "text": element.text if hasattr(element, "text") else "",
            })


        try:
            del converter, result_doc
            _ = gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                console.print("  [dim]CUDA cache cleared[/dim]")
        except ImportError:
            pass

        # Docling doesn't give us easy per-page text splits,
        # so we use the full markdown as a single "page" entry.
        # Cell 4 will work from raw_text directly for Docling.
        return ExtractionResult(
            parser_name=self.name,
            raw_text=markdown_text,
            pages=[markdown_text],
            metadata=metadata_elements,
        )

In [8]:
from unstructured.partition.pdf import partition_pdf

class UnstructuredParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:
        

        elements = partition_pdf(
            filename=str(pdf_path),
            strategy="fast",
            include_page_breaks=True,
        )

        pages = []
        metadata_elements = []

        # Elements we want to EXCLUDE from raw_text.
        # Headers and footers in this textbook are page numbers and
        # chapter titles repeated on every page — pure noise for RAG.

        for element in elements:
            category = element.category if hasattr(element, "category") else "Unknown"

            if category == "PageBreak":
                continue  # skip, we use metadata.page_number instead
            
            page_num = getattr(element.metadata, "page_number", 0) or 0

            metadata_elements.append({
                "category": category,
                "text": str(element),
                "page": page_num,
            })

            if category not in {"Header", "Footer"}:
                # Grow pages list dynamically using actual page numbers
                while len(pages) < page_num:
                    pages.append("")
                if page_num > 0:
                    pages[page_num - 1] += "\n" + str(element)

        raw_text = "\n".join(pages)

        return ExtractionResult(
            parser_name=self.name,
            raw_text=raw_text,
            pages=pages,
            metadata=metadata_elements,
        )

In [9]:
import openparse

class OpenParseParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:
        

        parser = openparse.DocumentParser(
            table_args={
                "parsing_algorithm": "pymupdf",
            }
        )

        console.print("  [dim]Running openparse structural analysis...[/dim]")
        parsed_doc = parser.parse(str(pdf_path))

        pages = []
        metadata_nodes = []
        all_texts = []

        for node in tqdm(parsed_doc.nodes, desc="openparse nodes"):
            node_text = node.text if hasattr(node, "text") else str(node)
            all_texts.append(node_text)

            metadata_nodes.append(json.loads(node.model_dump_json()))

        # openparse doesn't give us page-level splits directly.
        # We use the full text as a single entry and rely on
        # node-level metadata for page attribution in Cell 4.
        raw_text = "\n\n".join(all_texts)
        pages = [raw_text]

        return ExtractionResult(
            parser_name=self.name,
            raw_text=raw_text,
            pages=pages,
            metadata=metadata_nodes,
        )

In [10]:
from liteparse import LiteParse

class LiteParseParser(BaseParser):

    def _extract(self, pdf_path: Path) -> ExtractionResult:

        console.print("  [dim]Running LiteParse spatial extraction...[/dim]")

        parser = LiteParse()
        result = parser.parse(str(pdf_path))

        # LiteParse returns spatial text — layout preserved via
        # indentation and whitespace, not Markdown fencing.
        # result.text is the full document text.
        # result.pages is a list of per-page results with
        # bounding box data per text item.
        full_text = result.text

        pages = [page.text for page in result.pages] if hasattr(result, "pages") else [full_text]

        metadata_entries = []
        if hasattr(result, "pages"):
            for i, page in enumerate(result.pages):
                metadata_entries.append({
                    "page": i,
                    "item_count": len(page.items) if hasattr(page, "items") else 0,
                })

        return ExtractionResult(
            parser_name=self.name,
            raw_text=full_text,
            pages=pages,
            metadata=metadata_entries,
        )

In [11]:
ALL_PARSERS: list[BaseParser] = [
    PyMuPDFParser(),
    PDFPlumberParser(),
    OpenParseParser(),
    UnstructuredParser(),
    DoclingParser(),
    LiteParseParser(),
]

# Filter to only parsers whose dependencies are installed
AVAILABLE_PARSERS = [p for p in ALL_PARSERS if p.is_available()]

console.print(f"\n[bold]Parser registry[/bold]")
for p in ALL_PARSERS:
    status = "[green]✓[/green]" if p.is_available() else "[red]✗ not available[/red]"
    console.print(f"  {status} {p.name}")

console.print(f"\n[bold]{len(AVAILABLE_PARSERS)}/{len(ALL_PARSERS)} parsers ready[/bold]")

Parser registry

✓ pymupdf

✓ pdfplumber

✓ openparse

✓ unstructured

✓ docling

✓ liteparse

6/6 parsers ready

In [12]:
import gc
import psutil

extraction_results: dict[str, ExtractionResult] = {}
GPU_PARSERS = {"docling"}

CPU_PARSERS_LIST = [p for p in AVAILABLE_PARSERS if p.name not in GPU_PARSERS]

ram = psutil.virtual_memory()
console.print(f"Available RAM: {ram.available / 1e9:.1f}GB / {ram.total / 1e9:.1f}GB\n")
console.print(f"[bold]Running CPU parsers on:[/bold] {PDF_PATH.name}\n")

assert PDF_PATH.exists(), f"PDF not found: {PDF_PATH}"

for parser in CPU_PARSERS_LIST:
    console.rule(f"[cyan]{parser.name}[/cyan]")
    result = parser.extract(PDF_PATH)
    extraction_results[parser.name] = result

    if result.error:
        console.print(f"  [red]✗ Failed: {result.error}[/red]")
    else:
        console.print(f"  [green]✓ Done[/green] — "
                      f"{len(result.pages)} pages, "
                      f"{len(result.raw_text):,} chars, "
                      f"{result.extraction_time_s:.1f}s")

# Free RAM before GPU parsers
gc.collect()
ram = psutil.virtual_memory()
console.print(f"\n[bold green]CPU parsers complete.[/bold green]")
console.print(f"Available RAM after cleanup: {ram.available / 1e9:.1f}GB / {ram.total / 1e9:.1f}GB")
console.print("Close any heavy applications before running Cell 3.2.")

Available RAM: 7.1GB / 16.6GB

Running CPU parsers on: Jeanne Boyarsky, Scott Selikoff - OCP Oracle Certified Professional Java SE 17 Developer 
Study Guide - Exam 1Z0-829.-Sybex (2022).pdf

───────────────────────────────────────────────────── pymupdf ─────────────────────────────────────────────────────

Cache hit for pymupdf — loading from pymupdf_d8a2f26d8e91af88.json

✓ Done — 1059 pages, 1,966,343 chars, 4.2s

...

Cache hit for liteparse — loading from liteparse_d8a2f26d8e91af88.json

✓ Done — 1059 pages, 2,148,888 chars, 111.1s

CPU parsers complete.

Available RAM after cleanup: 7.0GB / 16.6GB

Close any heavy applications before running Cell 3.2.

In [13]:
import gc
import psutil

GPU_PARSERS_LIST = [p for p in AVAILABLE_PARSERS if p.name in GPU_PARSERS]

ram = psutil.virtual_memory()
console.print(f"Available RAM: {ram.available / 1e9:.1f}GB / {ram.total / 1e9:.1f}GB\n")

if ram.available / 1e9 < 8:
    console.print(
        "[bold red]⚠ Less than 8GB RAM available. "
        "Risk of OOM crash. Close other applications before proceeding.[/bold red]"
    )

console.print(f"[bold]Running GPU parsers on:[/bold] {PDF_PATH.name}\n")

for parser in GPU_PARSERS_LIST:
    console.rule(f"[cyan]{parser.name}[/cyan]")
    result = parser.extract(PDF_PATH)
    extraction_results[parser.name] = result

    if result.error:
        console.print(f"  [red]✗ Failed: {result.error}[/red]")
    else:
        console.print(f"  [green]✓ Done[/green] — "
                      f"{len(result.pages)} pages, "
                      f"{len(result.raw_text):,} chars, "
                      f"{result.extraction_time_s:.1f}s")

    # Free VRAM + RAM after each GPU parser
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
            console.print(f"  [dim]VRAM cleared after {parser.name}[/dim]")
    except ImportError:
        pass
    gc.collect()
    ram = psutil.virtual_memory()
    console.print(f"  [dim]RAM after {parser.name}: {ram.available / 1e9:.1f}GB available[/dim]")

console.print("\n[bold green]GPU parsers complete.[/bold green]")

Available RAM: 7.0GB / 16.6GB

⚠ Less than 8GB RAM available. Risk of OOM crash. Close other applications before proceeding.

Running GPU parsers on: Jeanne Boyarsky, Scott Selikoff - OCP Oracle Certified Professional Java SE 17 Developer 
Study Guide - Exam 1Z0-829.-Sybex (2022).pdf

───────────────────────────────────────────────────── docling ─────────────────────────────────────────────────────

Cache hit for docling — loading from docling_d8a2f26d8e91af88.json

✓ Done — 1 pages, 2,233,974 chars, 568.8s

VRAM cleared after docling

RAM after docling: 7.0GB available

GPU parsers complete.

In [14]:
# ============================================================
# CELL 4.1 — Chunking Pipeline
#
# Runs all 6 parsers × 2 chunkers = 12 combinations.
# Produces all_chunks: dict keyed by (parser_name, chunker_name)
# → list[Chunk]
#
# Special handling:
# - Docling + openparse return a single large string.
#   We pre-split on Markdown headings before SemanticChunker
#   to avoid embedding thousands of sentences in one shot.
# - SemanticChunker is initialized once and reused.
# - RecursiveCharacterTextSplitter is stateless, no init cost.
# ============================================================

import re
import gc
from dataclasses import asdict

# ---- Initialize chunkers (once) ----

console.print("[bold]Initializing chunkers...[/bold]")


from chonkie import MarkdownChef

MARKDOWN_PARSERS = {"docling"}  # parsers that output Markdown
markdown_chef = MarkdownChef()

def chunk_markdown_parser(parser_name: str, raw_text: str) -> list[str]:
    """
    For Markdown-output parsers: use MarkdownChef to extract
    code blocks and tables as discrete chunks, then apply
    SemanticChunker only on prose text.
    """
    # Write to a temp file since MarkdownChef takes a file path
    import tempfile
    with tempfile.NamedTemporaryFile(
        mode="w", suffix=".md", delete=False, encoding="utf-8"
    ) as f:
        f.write(raw_text)
        tmp_path = f.name

    doc = markdown_chef.process(tmp_path)
    Path(tmp_path).unlink()  # clean up

    chunk_texts = []

    # Code blocks → each is its own chunk, no further splitting
    for code in doc.code:
        if code.content.strip():
            chunk_texts.append(code.content)

    # Tables → each is its own chunk
    for table in doc.tables:
        if table.content.strip():
            chunk_texts.append(table.content)

    # Prose text chunks → run through SemanticChunker in windows
    prose_text = "\n\n".join(c.text for c in doc.chunks if c.text.strip())
    if prose_text:
        windows = presplit_markdown(prose_text)
        for window in tqdm(windows, desc=f"  semantic chunks ({parser_name})"):
            if window.strip():
                semantic_chunks = semantic_chunker.chunk(window)
                chunk_texts.extend([c.text for c in semantic_chunks])

    return chunk_texts

semantic_chunker = SemanticChunker(
    embedding_model=EMBEDDING_MODEL_NAME,
    chunk_size=SEMANTIC_CHUNK_SIZE,        # 1024 tokens
    threshold=0.5,                         # same as CodeMentor production
    min_sentences_per_chunk=1,
    skip_window=1,                         # look past code blocks when merging
)
console.print(f"  ✓ SemanticChunker (model={EMBEDDING_MODEL_NAME}, chunk_size={SEMANTIC_CHUNK_SIZE})")

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=RECURSIVE_CHUNK_SIZE,
    chunk_overlap=RECURSIVE_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
console.print(f"  ✓ RecursiveCharacterTextSplitter (chunk_size={RECURSIVE_CHUNK_SIZE}, overlap={RECURSIVE_CHUNK_OVERLAP})")

# ---- Parsers that return a single large string ----
# These need pre-splitting before SemanticChunker
# to avoid OOM on embedding thousands of sentences at once.
SINGLE_PAGE_PARSERS = {"docling", "openparse"}

def presplit_markdown(text: str, window_size: int = 30000) -> list[str]:
    """
    Split a large Markdown string into windows at heading boundaries.
    Used for Docling and openparse output before SemanticChunker.

    We split on ## and ### headings because:
    - Docling uses these for chapter/section boundaries
    - openparse joins nodes with \n\n, so headings act as anchors
    - 30000 chars ≈ ~7500 tokens, well within SemanticChunker's range
    """
    # Split at any Markdown heading (# ## ###)
    sections = re.split(r'(?=^#{1,3} )', text, flags=re.MULTILINE)

    windows = []
    current = ""
    for section in sections:
        if not section.strip():
            continue
        if len(current) + len(section) > window_size:
            if current.strip():
                windows.append(current)
            current = section
        else:
            current += section
    if current.strip():
        windows.append(current)

    # Fallback: if no headings found, split by paragraph
    if not windows:
        paragraphs = text.split("\n\n")
        current = ""
        for para in paragraphs:
            if len(current) + len(para) > window_size:
                if current.strip():
                    windows.append(current)
                current = para
            else:
                current += "\n\n" + para
        if current.strip():
            windows.append(current)

    return windows


def chunk_with_semantic(parser_name: str, raw_text: str) -> list[str]:
    if parser_name in MARKDOWN_PARSERS:
        return chunk_markdown_parser(parser_name, raw_text)
    
    # All other parsers — window-based semantic chunking
    windows = presplit_markdown(raw_text)
    console.print(f"    [dim]Pre-split into {len(windows)} windows[/dim]")
    chunk_texts = []
    for window in tqdm(windows, desc=f"  semantic chunks ({parser_name})"):
        if window.strip():
            chunks = semantic_chunker.chunk(window)
            chunk_texts.extend([c.text for c in chunks])
    return chunk_texts


def chunk_with_recursive(raw_text: str) -> list[str]:
    """
    Chunk text using RecursiveCharacterTextSplitter.
    Stateless — no special handling needed for large strings.
    """
    return recursive_splitter.split_text(raw_text)


# ---- Run all 12 combinations ----

all_chunks: dict[tuple[str, str], list[Chunk]] = {}

for parser_name, result in extraction_results.items():
    if result.error:
        console.print(f"[yellow]Skipping {parser_name} — extraction failed[/yellow]")
        continue

    console.rule(f"[cyan]{parser_name}[/cyan]")

    for chunker_name in ["semantic", "recursive"]:
        console.print(f"\n  [bold]Chunker: {chunker_name}[/bold]")

        if chunker_name == "semantic":
            chunk_texts = chunk_with_semantic(parser_name, result.raw_text)
        else:
            chunk_texts = chunk_with_recursive(result.raw_text)

        # Build Chunk objects
        chunks = []
        for i, text in enumerate(chunk_texts):
            if not text.strip():
                continue
            chunk = Chunk(
                chunk_id=f"{parser_name}_{chunker_name}_{i:04d}",
                text=text,
                parser=parser_name,
                chunker=chunker_name,
                char_count=len(text),
                token_count=len(text) // 4,  # approximation
            )
            chunks.append(chunk)

        all_chunks[(parser_name, chunker_name)] = chunks
        console.print(f"  ✓ {len(chunks)} chunks produced")

    # Free memory between parsers
    gc.collect()

console.print("\n[bold green]Chunking pipeline complete.[/bold green]")
console.print(f"Total combinations: {len(all_chunks)}")
console.print(f"Total chunks across all combinations: {sum(len(v) for v in all_chunks.values()):,}")

Initializing chunkers...

/home/dali/pdf-parser-benchmark/.venv/lib/python3.12/site-packages/chonkie/embeddings/sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


✓ SemanticChunker (model=BAAI/bge-small-en-v1.5, chunk_size=1024)

✓ RecursiveCharacterTextSplitter (chunk_size=1000, overlap=150)

───────────────────────────────────────────────────── pymupdf ─────────────────────────────────────────────────────

...

Chunker: recursive

✓ 3005 chunks produced

Chunking pipeline complete.

Total combinations: 12

Total chunks across all combinations: 36,497

In [15]:
# ============================================================
# CELL 4.2 — Per-Chunk Quality Annotation
#
# Annotates each chunk with quality signals:
#   - starts_mid_sentence
#   - ends_mid_sentence
#   - contains_code_block
#   - code_block_fragmented
#
# These are heuristics — not perfect, but consistent across
# all parsers which is what matters for comparison.
# ============================================================

def annotate_chunk(chunk: Chunk) -> Chunk:
    """
    Annotates a chunk in-place with quality signals.
    Returns the same chunk for chaining.
    """
    text = chunk.text.strip()

    if not text:
        return chunk

    # ---- Boundary quality ----

    # starts_mid_sentence: first char is lowercase
    # Excludes code blocks (which legitimately start with lowercase)
    first_char = text[0]
    chunk.starts_mid_sentence = (
        first_char.islower()
        and not text.startswith("```")
        and not text.startswith("    ")  # indented code
    )

    # ends_mid_sentence: last non-whitespace char is not terminal punctuation
    last_char = text.rstrip()[-1] if text.rstrip() else ""
    terminal_punctuation = {".", "?", "!", ":", ";", "`", '"', "'", ")"}
    chunk.ends_mid_sentence = last_char not in terminal_punctuation

    # ---- Code quality ----

    chunk.contains_code_block = _heuristic_is_code(text)

    if chunk.contains_code_block:
        # Fragmentation check: count braces
        # An intact code block should have balanced braces
        open_braces = text.count("{")
        close_braces = text.count("}")

        # Allow 1 brace imbalance for single-line snippets
        # (e.g., a method signature without body)
        chunk.code_block_fragmented = abs(open_braces - close_braces) > 1

    return chunk


# Annotate all chunks in-place
total_annotated = 0
for (parser_name, chunker_name), chunks in all_chunks.items():
    for chunk in chunks:
        annotate_chunk(chunk)
    total_annotated += len(chunks)

console.print(f"[green]✓ Annotated {total_annotated:,} chunks[/green]")

✓ Annotated 36,497 chunks

In [16]:
# ============================================================
# CELL 4.3 — Aggregate Quality Reports
#
# Collapses per-chunk annotations into one ChunkQualityReport
# per (parser, chunker) pair.
# ============================================================

quality_reports: dict[tuple[str, str], ChunkQualityReport] = {}

for (parser_name, chunker_name), chunks in all_chunks.items():
    if not chunks:
        continue

    total = len(chunks)
    code_chunks = [c for c in chunks if c.contains_code_block]

    report = ChunkQualityReport(
        parser=parser_name,
        chunker=chunker_name,
        total_chunks=total,
        avg_chunk_size_chars=sum(c.char_count for c in chunks) / total,
        code_block_preservation_rate=(
            sum(1 for c in code_chunks if not c.code_block_fragmented) / len(code_chunks)
            if code_chunks else 0.0
        ),
        mid_sentence_start_rate=sum(1 for c in chunks if c.starts_mid_sentence) / total,
        mid_sentence_end_rate=sum(1 for c in chunks if c.ends_mid_sentence) / total,
        # avg_rouge_l filled in Cell 4.4
    )

    quality_reports[(parser_name, chunker_name)] = report

# Print summary table
from rich.table import Table as RichTable

table = RichTable(title="Chunk Quality Reports", show_lines=True)
table.add_column("Parser",         style="cyan",  no_wrap=True)
table.add_column("Chunker",        style="white")
table.add_column("Chunks",         justify="right")
table.add_column("Avg chars",      justify="right")
table.add_column("Code pres. %",   justify="right")
table.add_column("Mid-start %",    justify="right")
table.add_column("Mid-end %",      justify="right")

for (parser_name, chunker_name), r in quality_reports.items():
    table.add_row(
        parser_name,
        chunker_name,
        str(r.total_chunks),
        f"{r.avg_chunk_size_chars:.0f}",
        f"{r.code_block_preservation_rate * 100:.1f}%",
        f"{r.mid_sentence_start_rate * 100:.1f}%",
        f"{r.mid_sentence_end_rate * 100:.1f}%",
    )

console.print(table)

                                  Chunk Quality Reports                                   
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Parser       ┃ Chunker   ┃ Chunks ┃ Avg chars ┃ Code pres. % ┃ Mid-start % ┃ Mid-end % ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━┩
│ pymupdf      │ semantic  │   3473 │       566 │        90.1% │       32.4% │     47.0% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ pymupdf      │ recursive │   2667 │       804 │        86.4% │       38.5% │     63.6% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ pdfplumber   │ semantic  │   3426 │       558 │        92.3% │       32.9% │     44.0% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ pdfplumber   │ recursive │   2289 │       940 │        87.4% │       59.6% │     68.5% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ openparse    │ semantic  │   3132 │       636 │        88.0% │       34.1% │     43.4% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ openparse    │ recursive │   3078 │       671 │        88.2% │       31.0% │     47.4% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ unstructured │ semantic  │   2313 │       813 │        99.2% │       10.9% │     19.0% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ unstructured │ recursive │   2636 │       744 │        91.5% │       18.7% │     41.4% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ liteparse    │ semantic  │   3674 │       585 │        90.3% │       34.2% │     44.9% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ liteparse    │ recursive │   3103 │       724 │        90.9% │       22.8% │     53.5% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ docling      │ semantic  │   3701 │       600 │        92.5% │       23.8% │     41.5% │
├──────────────┼───────────┼────────┼───────────┼──────────────┼─────────────┼───────────┤
│ docling      │ recursive │   3005 │       756 │        90.4% │        1.8% │     41.1% │
└──────────────┴───────────┴────────┴───────────┴──────────────┴─────────────┴───────────┘

In [17]:
# ============================================================
# CELL 4.4 — ROUGE-L Against Reference Passages
#
# 8 manually selected reference passages from the OCP Java 17
# Study Guide. These cover the 4 content types we care about:
#   1. Prose — concept definitions
#   2. Code  — Java method/class examples
#   3. Table — structured reference data
#   4. Inline-code-heavy — sentences with backtick expressions
#
# For each (parser, chunker) pair we find the chunk with the
# highest ROUGE-L score against each reference passage, then
# average across all 8 passages.
#
# ROUGE-L measures longest common subsequence — it captures
# whether the passage text survived parsing intact, not just
# whether the semantics are similar.
# ============================================================

REFERENCE_PASSAGES = [
    # ── Prose ──────────────────────────────────────────────
    {
        "id": "prose_1",
        "type": "prose",
        "text": (
            "Java is an object-oriented language, which means it is organized around objects. "
            "An object is a specific instance of a class. A class is like a blueprint for creating objects."
        ),
    },
    {
        "id": "prose_2",
        "type": "prose",
        "text": (
            "An interface is an abstract data type that declares a list of abstract methods "
            "that any class implementing the interface must provide."
        ),
    },
    # ── Code ───────────────────────────────────────────────
    {
        "id": "code_1",
        "type": "code",
        "text": (
            "public class Animal {\n"
            "    private String name;\n"
            "    public Animal(String name) {\n"
            "        this.name = name;\n"
            "    }\n"
            "    public String getName() {\n"
            "        return name;\n"
            "    }\n"
            "}"
        ),
    },
    {
        "id": "code_2",
        "type": "code",
        "text": (
            "var list = new ArrayList<String>();\n"
            "list.add(\"Fluffy\");\n"
            "list.add(\"Webby\");\n"
            "System.out.println(list);"
        ),
    },
    # ── Table ──────────────────────────────────────────────
    {
        "id": "table_1",
        "type": "table",
        "text": (
            "Operator  Description\n"
            "++        Increment by 1\n"
            "--        Decrement by 1"
        ),
    },
    {
        "id": "table_2",
        "type": "table",
        "text": (
            "Access modifier  Description\n"
            "private          Only accessible within the class\n"
            "protected        Accessible within the package and subclasses\n"
            "public           Accessible from anywhere"
        ),
    },
    # ── Inline-code-heavy ──────────────────────────────────
    {
        "id": "inline_1",
        "type": "inline_code",
        "text": (
            "The instanceof operator can be used with pattern matching. "
            "Instead of writing if (obj instanceof String) { String s = (String) obj; }, "
            "you can write if (obj instanceof String s) and use s directly."
        ),
    },
    {
        "id": "inline_2",
        "type": "inline_code",
        "text": (
            "A functional interface is an interface with exactly one abstract method. "
            "The @FunctionalInterface annotation is optional but recommended. "
            "Examples include Comparator<T>, Predicate<T>, and Supplier<T>."
        ),
    },
]

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=False)

def best_rouge_l(chunks: list[Chunk], reference: str) -> float:
    """
    Find the highest ROUGE-L score between `reference` and
    any single chunk in the list.
    """
    best = 0.0
    for chunk in chunks:
        score = scorer.score(reference, chunk.text)["rougeL"].fmeasure
        if score > best:
            best = score
    return best


for (parser_name, chunker_name), chunks in all_chunks.items():
    if not chunks:
        continue

    scores = []
    for passage in REFERENCE_PASSAGES:
        score = best_rouge_l(chunks, passage["text"])
        scores.append(score)

    avg_rouge = sum(scores) / len(scores)
    quality_reports[(parser_name, chunker_name)].avg_rouge_l = avg_rouge

# Print updated table with ROUGE-L
table2 = RichTable(title="Chunk Quality — with ROUGE-L", show_lines=True)
table2.add_column("Parser",       style="cyan",  no_wrap=True)
table2.add_column("Chunker",      style="white")
table2.add_column("Chunks",       justify="right")
table2.add_column("Code pres. %", justify="right")
table2.add_column("Mid-start %",  justify="right")
table2.add_column("ROUGE-L",      justify="right")

for (parser_name, chunker_name), r in quality_reports.items():
    table2.add_row(
        parser_name,
        chunker_name,
        str(r.total_chunks),
        f"{r.code_block_preservation_rate * 100:.1f}%",
        f"{r.mid_sentence_start_rate * 100:.1f}%",
        f"{r.avg_rouge_l:.4f}",
    )

console.print(table2)

                        Chunk Quality — with ROUGE-L                        
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Parser       ┃ Chunker   ┃ Chunks ┃ Code pres. % ┃ Mid-start % ┃ ROUGE-L ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━┩
│ pymupdf      │ semantic  │   3473 │        90.1% │       32.4% │  0.2750 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ pymupdf      │ recursive │   2667 │        86.4% │       38.5% │  0.2686 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ pdfplumber   │ semantic  │   3426 │        92.3% │       32.9% │  0.2713 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ pdfplumber   │ recursive │   2289 │        87.4% │       59.6% │  0.2097 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ openparse    │ semantic  │   3132 │        88.0% │       34.1% │  0.2644 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ openparse    │ recursive │   3078 │        88.2% │       31.0% │  0.2531 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ unstructured │ semantic  │   2313 │        99.2% │       10.9% │  0.2459 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ unstructured │ recursive │   2636 │        91.5% │       18.7% │  0.2257 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ liteparse    │ semantic  │   3674 │        90.3% │       34.2% │  0.2878 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ liteparse    │ recursive │   3103 │        90.9% │       22.8% │  0.2392 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ docling      │ semantic  │   3701 │        92.5% │       23.8% │  0.3296 │
├──────────────┼───────────┼────────┼──────────────┼─────────────┼─────────┤
│ docling      │ recursive │   3005 │        90.4% │        1.8% │  0.2627 │
└──────────────┴───────────┴────────┴──────────────┴─────────────┴─────────┘